# Agent Runtime への ADK エージェントのデプロイ

## 概要

このノートブックでは、対話型 AI エージェントを **Agent Runtime** にデプロイして管理する方法について説明します。 

## 学習目標

このノートブックを完了すると、以下のことができるようになります。
* 必要な構成をセットアップし、ADK **Agents** を **Agent Runtime** にデプロイする。
* ADK の **RemoteAgent** と **RemoteSession** を使用して、エージェントとのインタラクションを実行する。
* Agent Platform SDK を使用して、Agent Runtime にデプロイしたエージェントを管理（一覧表示/更新）および削除する。

## ADK エージェントのデプロイオプション
ADK エージェントは、本番環境への対応やカスタムの柔軟性に関するニーズに応じて、さまざまな環境にデプロイできます。

#### Agent Platform の Agent Runtime
Agent Runtime は、ADK などのフレームワークで構築された AI エージェントのデプロイ、管理、スケーリング専用に設計された、Google Cloud 上のフルマネージド自動スケーリングサービスです。

[Agent Runtime へのエージェントのデプロイに関する詳細はこちらをご覧ください。](https://google.github.io/adk-docs/deploy/agent-engine/)

#### Cloud Run
Cloud Run は、コンテナベースのアプリケーションとしてエージェントを実行できる、Google Cloud 上のマネージド自動スケーリングコンピューティングプラットフォームです。

[Cloud Run へのエージェントのデプロイに関する詳細はこちらをご覧ください。](https://google.github.io/adk-docs/deploy/cloud-run/)

#### Google Kubernetes Engine（GKE）
Google Kubernetes Engine（GKE）は、コンテナ化された環境でエージェントを実行できる Google Cloud のマネージド Kubernetes サービスです。GKE は、デプロイをより細かく制御する必要がある場合や、オープンモデルを実行する場合に適した選択肢です。

[GKE へのエージェントのデプロイに関する詳細はこちらをご覧ください。](https://google.github.io/adk-docs/deploy/gke/)

### Agent Runtime の概要
**Agent Runtime** は、開発者が本番環境で AI エージェントをデプロイ、管理、スケーリングできるようにする一連のサービスです。本番環境でエージェントをスケーリングするためのインフラストラクチャを処理するため、アプリケーションの作成に集中できます。Agent Runtime は Agent Platform 内の Gemini モデル用 Python SDK と密接に統合されており、プロンプト、エージェント、サンプルをモジュール形式で管理できます。また、ADK、LangChain、LlamaIndex、その他の Python フレームワークとも互換性があります。

以下は、Agent Runtime が提供するサービスであり、個別に使用することも、組み合わせて使用することもできます。

**Runtime**: マネージドランタイムとエンドツーエンドの管理機能を使用して、エージェントをデプロイおよびスケーリングします。

**品質と評価（Quality and evaluation）**: 統合された Gen AI Evaluation サービスを使用してエージェントの品質を評価し、Gemini モデルのトレーニング実行によってエージェントを最適化します。

**Example Store**: few-shot のサンプルを保存および動的に取得して、エージェントのパフォーマンスを向上させます。

**Sessions**: Agent Runtime Sessions により、ユーザーとエージェント間の個々のインタラクションを保存し、会話コンテキストの信頼できるソースを提供します。

**Memory Bank**: Agent Runtime Memory Bank により、セッションから情報を保存および取得して、エージェントとのインタラクションをパーソナライズします。

## パッケージのインストール

In [ ]:
import importlib
import json
import os
import warnings

import vertexai
from vertexai.preview.reasoning_engines import AdkApp

# すべての警告を無視
warnings.filterwarnings("ignore")

import logging

logging.basicConfig(level=logging.ERROR)

In [ ]:
GEMINI_LOCATION = "global"
os.environ["GOOGLE_CLOUD_LOCATION"] = GEMINI_LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"  # Agent Platform API を使用

In [ ]:
%%bash
mkdir -p adk_agents/agent1_weather_lookup
echo > adk_agents/.env "GOOGLE_CLOUD_LOCATION=$GOOGLE_CLOUD_LOCATION
GOOGLE_GENAI_USE_VERTEXAI=$GOOGLE_GENAI_USE_VERTEXAI
"

## エージェントファイルの準備

ここでは、[building_agent_with_adk.ipynb](../../building_agents/solutions/building_agent_with_adk.ipynb) ノートブックで作成した基本的なエージェントファイルを再利用します。

まだそのノートブックを実行していない場合は、以下のセルを実行してファイルを作成してください。

In [ ]:
%%writefile ./adk_agents/agent1_weather_lookup/tools.py
def get_weather(city: str) -> dict:
    """指定された都市の現在の気象情報を取得します。

    Args:
        city (str): 都市の名前（例: "New York", "London", "Tokyo"）。

    Returns:
        dict: 気象情報を含む辞書。
              'status' キー（'success' または 'error'）を含みます。
              'success' の場合は、気象情報の詳細を含む 'report' キーが含まれます。
              'error' の場合は、'error_message' キーが含まれます。
    """
    print(f"--- Tool: get_weather called for city: {city} ---") # ツールの実行をログに記録
    city_normalized = city.lower().replace(" ", "") # 基本的な正規化

    # モックの気象データ
    mock_weather_db = {
        "newyork": {"status": "success", "report": "The weather in New York is sunny with a temperature of 25°C."},
        "london": {"status": "success", "report": "It's cloudy in London with a temperature of 15°C."},
        "tokyo": {"status": "success", "report": "Tokyo is experiencing light rain and a temperature of 18°C."},
    }

    if city_normalized in mock_weather_db:
        return mock_weather_db[city_normalized]
    else:
        return {"status": "error", "error_message": f"Sorry, I don't have weather information for '{city}'."}

In [ ]:
%%writefile ./adk_agents/agent1_weather_lookup/agent.py
from google.adk.agents import Agent
MODEL = "gemini-3.5-flash"

from .tools import get_weather

root_agent = Agent(
    name="weather_agent_v1",
    model=MODEL, # Gemini の文字列または LiteLlm オブジェクトを指定可能
    description="Provides weather information for specific cities.",
    instruction="You are a helpful weather assistant. "
                "When the user asks for the weather in a specific city, "
                "use the 'get_weather' tool to find the information. "
                "If the tool returns an error, inform the user politely. "
                "If the tool is successful, present the weather report clearly.",
    tools=[get_weather], # 関数を直接渡す
)

In [ ]:
APP_NAME = "weather_info_app"
USER_ID = "user_1"

## Agent Runtime へのデプロイ
Agent Runtime は、開発者が本番環境で AI エージェントをデプロイ、管理、スケーリングできるようにするフルマネージドの Google Cloud サービスです。 
Agent Runtime が本番環境でエージェントをスケーリングするためのインフラストラクチャを処理するため、インテリジェントで影響力のあるアプリケーションの作成に集中できます。

In [ ]:
LOCATION = "us-central1"
PROJECT = !gcloud config get-value project
PROJECT = PROJECT[0]
BUCKET_NAME = f"agent-deployment-{PROJECT}-bucket"
BUCKET_URI = f"gs://{BUCKET_NAME}"

**BUCKET の存在を確認します。存在しない場合は作成します:**

In [ ]:
!gcloud storage ls --buckets | grep -w $BUCKET_URI || gcloud storage buckets create --location=$LOCATION $BUCKET_URI

### 初期化

In [ ]:
client = vertexai.Client(project=PROJECT, location=LOCATION)

### Agent Runtime 向けにエージェントを準備する
`reasoning_engines.AdkApp()` を使用してエージェントをラップし、Agent Runtime にデプロイできるようにします。

In [ ]:
from adk_agents.agent1_weather_lookup import agent

importlib.reload(agent)
adk_app = AdkApp(agent=agent.root_agent)

### エージェントを Agent Runtime にデプロイする

これで、以下を指定して `client.agent_engines.create()` を呼び出すことで、Agent Platform の Agent Runtime にエージェントをデプロイする準備が整いました。

1. エージェントクラスのインスタンス
2. `requirements.txt` ファイルでパッケージとバージョンを定義するのと同様に、実行時にエージェントが必要とする Python パッケージとバージョン

この手順には数分かかる場合があります。

In [ ]:
DISPLAY_NAME = agent.root_agent.name
env_vars = {
    "GOOGLE_CLOUD_LOCATION": GEMINI_LOCATION,
    "GOOGLE_GENAI_USE_VERTEXAI": "true",
    "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "true",
    "OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT": "true",
}

remote_app = client.agent_engines.create(
    agent=adk_app,
    config={
        "display_name": DISPLAY_NAME,
        "description": "Weather Agent v1",
        "requirements": [
            "google-adk==2.5.0",
            "google-genai==2.11.0",
            "google-cloud-aiplatform[agent_engines,adk]",
        ],
        "extra_packages": ["./adk_agents/"],
        "env_vars": env_vars,
        "staging_bucket": BUCKET_URI,
    },
)

print(f"Created remote agent: {remote_app.api_resource.display_name}")

デプロイには数分かかります。その間、バックグラウンドで次の手順が実行されます。

ローカルで以下のアーティファクトのバンドルが生成されます。
 - *.pkl: local_agent に対応する pickle ファイル。
 - requirements.txt: パッケージの要件を含むテキストファイル。
 - dependencies.tar.gz: 追加のパッケージを含む tar ファイル。

アーティファクトをステージングするために、バンドルが Cloud Storage（対応するフォルダ配下）にアップロードされます。
各アーティファクトの Cloud Storage URI は PackageSpec で指定されます。
Agent Runtime サービスがリクエストを受信し、バックエンドでコンテナをビルドして HTTP サーバーを起動します。

デプロイのレイテンシは、必要なパッケージのインストールにかかる合計時間に依存します。
デプロイが完了すると、remote_agent は Agent Platform 上で実行されているエージェントのインスタンスに対応し、クエリの送信や削除が可能になります。
デプロイされたエージェントは、Cloud Console（Agent Platform -> Agent Runtime）で確認できます。

[Agent Runtime コンソール](https://console.cloud.google.com/agent-platform/runtimes)

### Agent Runtime 上で ADK エージェントを試す
#### セッションの作成（リモート）
テキストメッセージを毎回ゼロから始めないのと同様に、エージェントも進行中のインタラクションに関するコンテキストを必要とします。
Session は、これらの個々の会話スレッドを追跡および管理するために特別に設計された ADK オブジェクトです。
ユーザーがエージェントとのインタラクションを開始すると、SessionService はこのユーザーの進行中のインタラクションを追跡するための Session オブジェクトを作成します。
**ここで USER_ID は、128 文字以内のユーザー定義 ID です。**

In [ ]:
remote_session = await remote_app.async_create_session(user_id=USER_ID)

#### create_session（リモート）のセッションプロパティ出力の確認:
セッションは、ユーザーと AI エージェント間のインタラクションを表します。会話履歴、エージェントのアクション、メモリを追跡し、エージェントがインタラクション間でコンテキストを維持できるようにします。
セッション情報を見てみましょう。

In [ ]:
remote_session

#### エージェントへのクエリ送信（リモート）
生の ADK エージェント出力を処理するヘルパー関数 `print_adk_output(event)` を定義しましょう。
この関数は、ツール呼び出し、ツールのレスポンス、最終的なテキスト出力の各ステップを識別し、わかりやすくフォーマットします。

In [ ]:
def print_adk_output(event: dict):
    # コンテンツの最初のパートを抽出
    part = event.get("content", {}).get("parts", [{}])[0]
    author = event.get("author", "unknown_agent")
    print(f"\n--- [Event: {author}] ---")

    # ケース 1: モデルが関数/ツールを呼び出している場合
    if "function_call" in part:
        call = part["function_call"]
        func_name = call.get("name", "N/A")
        func_args = call.get("args", {})
        print("Tool Call:")
        print(f"  - Function: {func_name}")
        print(f"  - Arguments: {json.dumps(func_args, indent=2)}")

    # ケース 2: ツールがレスポンスを返している場合
    elif "function_response" in part:
        response = part["function_response"]
        func_name = response.get("name", "N/A")
        func_response = response.get("response", {})
        print("Tool Response:")
        print(f"  - From: {func_name}")
        # レスポンス JSON を整形して出力
        print(f"  - Data: {json.dumps(func_response, indent=2)}")

    # ケース 3: モデルが最終的なテキストレスポンスを生成している場合
    elif "text" in part:
        text_response = part["text"].strip()
        print("Final Agent Response:")
        print(f"  -> {text_response}")

    else:
        print("Unknown step type.")

これで、.stream_query を使用してリモートエージェントにプロンプトを送信し、レスポンスを取得できるようになりました。

In [ ]:
async for event in remote_app.async_stream_query(
    user_id=USER_ID,
    session_id=remote_session["id"],
    message="What's the weather in New York?",
):
    print_adk_output(event)

#### エージェントのトレースの検査
トレースは、エージェントが各クエリに応答する際のリクエストのタイムラインです。
[Agent Runtime コンソール](https://console.cloud.google.com/agent-platform/runtimes) に移動し、デプロイされたエージェントを選択して [Trace] を開きます。

右上の「Graph」ビューと「Timeline」ビューを切り替えることができます。タイムラインビューでは、トレースは個々のスパンで構成されます。スパンは関数呼び出しや LLM とのインタラクションなどの単一の作業単位を表し、最初のスパンはリクエスト全体を表します。各スパンは、リクエスト内での特定の操作に関する詳細（操作名、開始時刻と終了時刻、関連する属性など）を提供します。

#### セッションの一覧表示（リモート）
Agent Platform Sessions は、ユーザーとエージェント間のインタラクションの履歴を保持します。
セッションは、長期記憶と会話コンテキストの信頼できるソースを提供します。
既存の会話を再開する場合は、特定のユーザーの既存の全セッションを取得できます。

In [ ]:
await remote_app.async_list_sessions(user_id=USER_ID)

Google Cloud コンソールを使用して既存のセッションを確認できます。
Google Cloud コンソールの Agent Platform に移動し、Agent Runtime に移動して、特定のエージェントの [Sessions] タブを開きます。

#### 特定のセッションの取得（リモート）
特定のエージェントが中断したところから再開できるように、（ID を使用して）特定のセッションを取得します。
**特定のセッションを取得するには、ユーザー ID とセッション ID の両方が必要です:**
ローカルでエージェントを使用している場合、セッション ID は `session.id` に格納されますが、Agent Runtime 上でリモートでエージェントを使用している場合、セッション ID は `remote_session["id"]` に格納されます。

In [ ]:
await remote_app.async_get_session(
    user_id=USER_ID, session_id=remote_session["id"]
)

### デプロイされたエージェントの管理
デプロイされたエージェントは、Agent Platform における reasoningEngine タイプのクラウドリソースです。

#### 指定したプロジェクトとロケーションのアクティブな全デプロイ済みエージェントを一覧表示する:
すべてのアクティブな AI エージェントのデプロイを一覧表示する必要がある場合は、`client.agent_engines.list()` メソッドを使用できます。
デプロイされた各エージェントには、一意の識別子 ***Name*** と完全修飾リソース名 ***Resource Name*** があります。

In [ ]:
agents_list = client.agent_engines.list()
for remote_agent in agents_list:
    print("\n------")
    print(f"Display Name: {remote_agent.api_resource.display_name}")
    print(f"Name: {remote_agent.api_resource.name}")
    print(f"Create Time: {remote_agent.api_resource.create_time}")

### デプロイされたエージェントの更新
非常に一般的なユースケースとして、新機能やバグ修正が含まれる可能性のある新しいバージョンでデプロイ済みエージェントを更新することが挙げられます。

#### 更新されたエージェントファイルの準備
ここでは、[building_agent_with_adk.ipynb](../../building_agents/solutions/building_agent_with_adk.ipynb) ノートブックで作成した基本的なエージェントファイルを再利用します。

まだそのノートブックを実行していない場合は、以下のセルを実行してファイルを作成してください。

In [ ]:
!mkdir -p adk_agents/agent2_sub_agent
!cp ./adk_agents/agent1_weather_lookup/tools.py ./adk_agents/agent2_sub_agent/

In [ ]:
%%writefile -a ./adk_agents/agent2_sub_agent/tools.py

def say_hello(name: str = "there") -> str:
    """簡単な挨拶を返します。オプションでユーザーの名前で呼びかけます。

    Args:
        name (str, optional): 挨拶する相手の名前。デフォルトは "there"。

    Returns:
        str: フレンドリーな挨拶メッセージ。
    """
    if name is None or name.strip() == "":
        name = "there"
    print(f"--- Tool: say_hello called with name: {name} ---")
    return f"Hello, {name}!"

def say_goodbye() -> str:
    """会話を締めくくるための簡単な別れのメッセージを返します。"""
    print(f"--- Tool: say_goodbye called ---")
    return "Goodbye! Have a great day."

In [ ]:
%%writefile ./adk_agents/agent2_sub_agent/agent.py
from google.adk.agents import Agent
MODEL = "gemini-3.5-flash"

from .tools import get_weather, say_hello, say_goodbye

# --- Greeting Agent ---
greeting_agent = Agent(
    model=MODEL,
    name="greeting_agent",
    instruction="You are the Greeting Agent. Your ONLY task is to provide a friendly greeting to the user. "
                "Use the 'say_hello' tool to generate the greeting. "
                "If the user provides their name, make sure to pass it to the tool. "
                "Do not engage in any other conversation or tasks.",
    description="Handles simple greetings and hellos using the 'say_hello' tool.", # 委任に不可欠
    tools=[say_hello],
)

In [ ]:
%%writefile -a ./adk_agents/agent2_sub_agent/agent.py

# --- Farewell Agent ---
farewell_agent = Agent(
    model=MODEL,
    name="farewell_agent",
    instruction="You are the Farewell Agent. Your ONLY task is to provide a polite goodbye message. "
                "Use the 'say_goodbye' tool when the user indicates they are leaving or ending the conversation "
                "(e.g., using words like 'bye', 'goodbye', 'thanks bye', 'see you'). "
                "Do not perform any other actions.",
    description="Handles simple farewells and goodbyes using the 'say_goodbye' tool.", # 委任に不可欠
    tools=[say_goodbye],
)

In [ ]:
%%writefile -a ./adk_agents/agent2_sub_agent/agent.py

root_agent = Agent(
    name="weather_agent_v2", # 新しいバージョン名を付与
    model=MODEL,
    description="The main coordinator agent. Handles weather requests and delegates greetings/farewells to specialists.",
    instruction="You are the main Weather Agent coordinating a team. Your primary responsibility is to provide weather information. "
                "Use the 'get_weather' tool ONLY for specific weather requests (e.g., 'weather in London'). "
                "You have specialized sub-agents: "
                "1. 'greeting_agent': Handles simple greetings like 'Hi', 'Hello'. Delegate to it for these. "
                "2. 'farewell_agent': Handles simple farewells like 'Bye', 'See you'. Delegate to it for these. "
                "Analyze the user's query. If it's a greeting, delegate to 'greeting_agent'. If it's a farewell, delegate to 'farewell_agent'. "
                "If it's a weather request, handle it yourself using 'get_weather'. "
                "For anything else, respond appropriately or state you cannot handle it.",
    tools=[get_weather],
    sub_agents=[greeting_agent, farewell_agent]
)

#### デプロイされたエージェントを更新後のエージェントに更新する
デプロイされたエージェントの 1 つ以上のフィールドを同時に更新できますが、更新するフィールドを少なくとも 1 つ指定する必要があります。デプロイされたエージェントの更新にかかる時間は実行される更新内容によって異なりますが、通常は数秒から数分かかります。

デプロイされたエージェント（***weather_agent_v1***）を更新後のエージェント（***weather_agent_v2***）に更新するには、次のようにします。

In [ ]:
from adk_agents.agent2_sub_agent import agent as agent2

importlib.reload(agent2)  # 強制リロード

DISPLAY_NAME = agent2.root_agent.name

adk_app_v2 = AdkApp(agent=agent2.root_agent)

remote_app = client.agent_engines.update(
    name=remote_app.api_resource.name,
    agent=adk_app_v2,
    config={
        "display_name": DISPLAY_NAME,
        "description": "Weather Agent v2",
        "requirements": [
            "google-adk==2.5.0",
            "google-genai==2.11.0",
            "google-cloud-aiplatform[agent_engines,adk]",
        ],
        "extra_packages": ["./adk_agents/"],
        "env_vars": env_vars,
        "staging_bucket": BUCKET_URI,
    },
)

デプロイの更新が反映されていることを確認するために、新しいセッションを作成します。

In [ ]:
remote_session = await remote_app.async_create_session(user_id=USER_ID)

エージェントの更新後に、変更された結果を確認します:

In [ ]:
async for event in remote_app.async_stream_query(
    user_id=USER_ID,
    session_id=remote_session["id"],
    message="Hello!",
):
    print_adk_output(event)

In [ ]:
async for event in remote_app.async_stream_query(
    user_id=USER_ID,
    session_id=remote_session["id"],
    message="Bye!",
):
    print_adk_output(event)

## 長期記憶（Long Term Memory）の使用
ここまで、Session が単一の進行中の会話の履歴（イベント）をどのように追跡するかを見てきました。しかし、エージェントが別のセッションでの過去の会話から情報を思い出す必要がある場合はどうすればよいでしょうか？ここで、長期ナレッジと MemoryService の概念が登場します。

Agent Runtime には組み込みの長期記憶サービスがあり、すでに有効になっています。ただし、その機能を連携させるには、メモリを生成および取得するためのツールやコールバックをいくつか追加する必要があります。

引き続き天気予報エージェントを使用し、それにメモリ機能を追加しましょう。

In [ ]:
!mkdir -p ./adk_agents/agent2_sub_agent_with_memory
!cp -r ./adk_agents/agent2_sub_agent/* ./adk_agents/agent2_sub_agent_with_memory

`agent.py` を上書きしましょう。ただし前半部分は同じです。

In [ ]:
%%writefile ./adk_agents/agent2_sub_agent_with_memory/agent.py
from google.adk.agents import Agent
from google import adk
MODEL = "gemini-3.5-flash"

from .tools import get_weather, say_hello, say_goodbye

# --- Greeting Agent ---
greeting_agent = Agent(
    model=MODEL,
    name="greeting_agent",
    instruction="You are the Greeting Agent. Your ONLY task is to provide a friendly greeting to the user. "
                "Use the 'say_hello' tool to generate the greeting. "
                "If the user provides their name, make sure to pass it to the tool. "
                "Do not engage in any other conversation or tasks.",
    description="Handles simple greetings and hellos using the 'say_hello' tool.", # 委任に不可欠
    tools=[say_hello],
)

# --- Farewell Agent ---
farewell_agent = Agent(
    model=MODEL,
    name="farewell_agent",
    instruction="You are the Farewell Agent. Your ONLY task is to provide a polite goodbye message. "
                "Use the 'say_goodbye' tool when the user indicates they are leaving or ending the conversation "
                "(e.g., using words like 'bye', 'goodbye', 'thanks bye', 'see you'). "
                "Do not perform any other actions.",
    description="Handles simple farewells and goodbyes using the 'say_goodbye' tool.", # 委任に不可欠
    tools=[say_goodbye],
)

それでは、メモリ機能を持つメインの `root_agent` を定義しましょう。

まず、必要に応じてメモリを生成および保存するツール（`auto_save_session_to_memory_callback`）を定義します。

取得に関して、ADK にはメモリを取得するための 2 つの事前構築済みツールが含まれています。
- `preload_memory`: 各ターンの開始時に常にメモリを取得し、プロンプトにアタッチします（コールバックに類似）。
- `load_memory`: エージェントが役立つと判断したときにメモリを取得します。

単純なユースケースには `preload_memory` で十分ですが、ここではメモリの取得がどのように機能するかを確認するために `load_memory` 戦略を試してみましょう。

In [ ]:
%%writefile ./adk_agents/agent2_sub_agent_with_memory/agent.py -a

async def auto_save_session_to_memory_callback(callback_context):
    await callback_context._invocation_context.memory_service.add_session_to_memory(
        callback_context._invocation_context.session)

root_agent = Agent(
    model=MODEL,
    name="memory_weather_agent_v3",
    description="The main coordinator agent. Handles weather requests and delegates greetings/farewells to specialists.",
    instruction="You are the main Weather Agent coordinating a team. Your primary responsibility is to provide weather information. "
                "Use the 'get_weather' tool ONLY for specific weather requests (e.g., 'weather in London'). "
                "You have specialized sub-agents: "
                "1. 'greeting_agent': Handles simple greetings like 'Hi', 'Hello'. Delegate to it for these. "
                "2. 'farewell_agent': Handles simple farewells like 'Bye', 'See you'. Delegate to it for these. "
                "Analyze the user's query. If it's a greeting, delegate to 'greeting_agent'. If it's a farewell, delegate to 'farewell_agent'. "
                "If it's a weather request, handle it yourself using 'get_weather'. "
                "For anything else, respond appropriately or state you cannot handle it."
                "Please use long term memory to personalize user experience.",
    # load_memory ツールを追加
    tools=[get_weather, adk.tools.load_memory],
    # auto_save_session_to_memory_callback を追加
    after_agent_callback=auto_save_session_to_memory_callback,
    sub_agents=[greeting_agent, farewell_agent]
)

エージェントを更新しましょう。

In [ ]:
from adk_agents.agent2_sub_agent_with_memory import agent as memory_agent

importlib.reload(memory_agent)  # 強制リロード

adk_app_memory = AdkApp(agent=memory_agent.root_agent)
DISPLAY_NAME = memory_agent.root_agent.name

remote_app = client.agent_engines.update(
    name=remote_app.api_resource.name,
    agent=adk_app_memory,
    config={
        "display_name": DISPLAY_NAME,
        "description": "Memory Weather Agent",
        "requirements": [
            "google-adk==2.5.0",
            "google-genai==2.11.0",
            "google-cloud-aiplatform[agent_engines,adk]",
        ],
        "extra_packages": ["./adk_agents/"],
        "env_vars": env_vars,
        "staging_bucket": BUCKET_URI,
    },
)

### メモリのテスト

In [ ]:
remote_session = await remote_app.async_create_session(user_id=USER_ID)

In [ ]:
async for event in remote_app.async_stream_query(
    user_id=USER_ID,
    session_id=remote_session["id"],
    message="What's the weather in New York?",
):
    print_adk_output(event)

将来パーソナライズに使用するために、エージェントに記憶させたい情報を入力してみましょう。

In [ ]:
async for event in remote_app.async_stream_query(
    user_id=USER_ID,
    session_id=remote_session["id"],
    message="Great to hear it is pretty warm! Actually I live in New York.",
):
    print_adk_output(event)

[Agent Runtime コンソール](https://console.cloud.google.com/agent-platform/runtimes) に移動し、[エージェントを選択] -> [Memories] で、情報が長期記憶として保存されているか確認します。

保存されていることが確認できたら、**別のセッションで** エージェントがそれを取得できるか確認しましょう！

In [ ]:
remote_session = await remote_app.async_create_session(user_id=USER_ID)

In [ ]:
async for event in remote_app.async_stream_query(
    user_id=USER_ID,
    session_id=remote_session["id"],
    message="What's the weather of the city I live in?",
):
    print_adk_output(event)

レスポンスをパーソナライズするために `load_memory` 関数が適切に呼び出されていることが確認できましたか？

## ボーナス: デプロイされたエージェントを Gemini Enterprise と統合する

エージェントを Agent Platform の Agent Runtime にデプロイしたら、Gemini Enterprise Agent Platform と統合して、Gemini Enterprise UI を介してユーザーが利用できるようにすることができます。

### 統合の手順:

1. **Gemini Enterprise アプリ（インスタンス）の作成**:
   - Google Cloud コンソールを開きます。
   - **Gemini Enterprise** ページ（または Search and Conversation）を検索して移動します。
   - **[Create Your First App]**（最初のアプリを作成）をクリックします。
   - （省略可）アプリの名前を入力し、適切なロケーションを選択します。
   - **[Create]**（作成）をクリックします。

2. **デプロイされたエージェントのリソース名の取得**:
   - Agent Platform -> [Agents] -> [Deployments] に移動します。
   - デプロイされたエージェントのリソース名を確認します。通常は `projects/{PROJECT_ID}/locations/{LOCATION}/reasoningEngines/{AGENT_ID}` のようになります。
   - これは、デプロイセルの出力またはエージェントを一覧表示することで確認できます。

3. **Gemini Enterprise へのエージェントの登録**:
   - Gemini Enterprise アプリで、**[Agents]** セクションまたは **[Agent Gallery]** に移動します。
   - **[Add Agent]**（エージェントを追加）をクリックします。
   - ソースとして **[Custom agent via Agent Engine]** を選択します。
   - **[Next]**（次へ）をクリックします。
   - エージェント名を追加します（例: `Weather Agent`）。
   - エージェントの説明を追加します（例: `Weather forecasting agent`）。
   - 上記で取得したエージェントのリソース名をコピーして、**[Agent Engine reasoning engine]** セクションに貼り付けます。
   - **[Create]**（作成）をクリックします。

4. **確認とテスト**:
   - Gemini Enterprise の概要ページに移動します。
   - **[Open Preview]**（プレビューを開く）をクリックします。
   - **[Agents]** -> 作成したエージェント名をクリックします。
   - カスタムエージェントとの会話をテストします。

### クリーンアップ
完了したら、クラウドリソースをクリーンアップすることをお勧めします。 
以下のコードを使用して、デプロイされた Agent Runtime インスタンスを削除できます。

In [ ]:
# remote_app.delete(force=True)

Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at

https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.